<a href="https://colab.research.google.com/github/MKhamK/README.md/blob/main/E_Grocery_Inventory_Analytics_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# ===== STEP 0: Import libraries =====
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [22]:

# ===== STEP 1: Load data =====
from google.colab import files
uploaded = files.upload()

Saving Inventory Management E-Grocery - InventoryData.csv to Inventory Management E-Grocery - InventoryData (1).csv


In [23]:
# แก้ชื่อไฟล์ตรงนี้
df = pd.read_csv('Inventory Management E-Grocery - InventoryData.csv')

In [24]:
# 2.1 ขนาดข้อมูล
print("1. Shape:", df.shape)

1. Shape: (1000, 37)


In [25]:
# 2.2 ตัวอย่าง 5 แถวแรก
print("\n2. First 5 rows:")
print(df.head())


2. First 5 rows:
    SKU_ID              SKU_Name       Category ABC_Class Supplier_ID  \
0  SKU0001     Pantry Product 13         Pantry         A        S005   
1  SKU0002     Fresh Product 112  Fresh Produce         C        S004   
2  SKU0003      Meat Product 446           Meat         B        S001   
3  SKU0004    Seafood Product 48        Seafood         A        S007   
4  SKU0005  Personal Product 194  Personal Care         A        S002   

           Supplier_Name Warehouse_ID    Warehouse_Location   Batch_ID  \
0           PT Agro Raya        WHBDG   Bandung - Rancaekek  BATCH2679   
1  PT Nusantara Supplier        WHDPS    Denpasar - Tabanan  BATCH4257   
2        PT Segar Makmur        WHBDG   Bandung - Rancaekek  BATCH6574   
3           PT Bakerindo        WHJKT  Jakarta - Cengkareng  BATCH5333   
4          PT Indo Fresh        WHDPS    Denpasar - Tabanan  BATCH6925   

  Received_Date  ... SKU_Churn_Rate Order_Frequency_per_month  \
0    2025-07-14  ...           2,

In [26]:
# 2.3 ชนิดข้อมูล
print("\n3. Data types:")
print(df.dtypes)

# 2.4 ค่าที่หายไป
print("\n4. Missing values:")
print(df.isnull().sum())

# 2.5 สถิติพื้นฐาน
print("\n5. Summary statistics:")
print(df.describe())



3. Data types:
SKU_ID                           object
SKU_Name                         object
Category                         object
ABC_Class                        object
Supplier_ID                      object
Supplier_Name                    object
Warehouse_ID                     object
Warehouse_Location               object
Batch_ID                         object
Received_Date                    object
Last_Purchase_Date               object
Expiry_Date                      object
Stock_Age_Days                    int64
Quantity_On_Hand                  int64
Quantity_Reserved                 int64
Quantity_Committed                int64
Damaged_Qty                       int64
Returns_Qty                       int64
Avg_Daily_Sales                  object
Forecast_Next_30d               float64
Days_of_Inventory                object
Reorder_Point                   float64
Safety_Stock                      int64
Lead_Time_Days                    int64
Unit_Cost_USD           

In [27]:
# 3.1 แปลงวันที่
date_cols = ['Received_Date','Last_Purchase_Date','Expiry_Date','Audit_Date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')


# 3.2 แปลงราคา US format
dollar_cols = ['Unit_Cost_USD','Last_Purchase_Price_USD','Total_Inventory_Value_USD']
for col in dollar_cols:
    df[col] = df[col].astype(str).str.replace('$', '').str.replace(',', '').astype(float)


# 3.3 แปลงตัวเลข EU format
number_cols = ['Avg_Daily_Sales','Days_of_Inventory','SKU_Churn_Rate','Order_Frequency_per_month']
for col in number_cols:
    df[col] = df[col].astype(str).str.replace('.', '').str.replace(',', '.').astype(float)


# 3.4 แปลงเปอร์เซ็นต์
percent_cols = ['Supplier_OnTime_Pct','Audit_Variance_Pct','Demand_Forecast_Accuracy_Pct']
for col in percent_cols:
    df[col] = df[col].astype(str).str.replace('%', '').str.replace(',', '.').astype(float)


print("✅ Cleaning complete!")

✅ Cleaning complete!


In [28]:
print(df['Avg_Daily_Sales'].head())

0    28.57
1    34.99
2    36.55
3    25.49
4    17.05
Name: Avg_Daily_Sales, dtype: float64


In [29]:
# ========================================
# 🎯 KPI 1: Median Sellable Coverage
# ========================================
# คำถาม: สินค้าขายได้อีกกี่วันก่อนหมด?
# สูตร: (Quantity_On_Hand - Reserved - Committed) / Avg_Daily_Sales

# ⚠️ เปลี่ยนชื่อคอลัมน์ให้ตรงกับ dataset ของคุณ
QTY_ON_HAND = 'Quantity_On_Hand'
QTY_RESERVED = 'Quantity_Reserved'
QTY_COMMITTED = 'Quantity_Committed'
DAILY_SALES = 'Avg_Daily_Sales'

# คำนวณ ATP (Available-to-Promise)
df['ATP'] = df[QTY_ON_HAND] - df[QTY_RESERVED] - df[QTY_COMMITTED]

# คำนวณจำนวนวันที่ขายได้
df['Sellable_Coverage_Days'] = df['ATP'] / df[DAILY_SALES]

# แสดงผล
median_coverage = df['Sellable_Coverage_Days'].median()
print(f"🎯 Median Sellable Coverage: {median_coverage:.1f} วัน")
print(f"   Target: 7-14 วัน")

🎯 Median Sellable Coverage: 6.2 วัน
   Target: 7-14 วัน


In [30]:
# ========================================
# 🎯 KPI 2: Value Expiring within 30 Days
# ========================================
# คำถาม: มูลค่าสินค้าที่จะหมดอายุใน 30 วันเท่าไร?

EXPIRY_DATE = 'Expiry_Date'
INVENTORY_VALUE = 'Total_Inventory_Value_USD'

# คำนวณวันก่อนหมดอายุ
today = pd.Timestamp.today()
df['Days_To_Expiry'] = (df[EXPIRY_DATE] - today).dt.days

# สินค้าที่จะหมดอายุใน 30 วัน
expiring_soon = df[df['Days_To_Expiry'] <= 30][INVENTORY_VALUE].sum()
total_value = df[INVENTORY_VALUE].sum()
expiring_pct = (expiring_soon / total_value) * 100

print(f"🎯 Value Expiring in 30 Days: ${expiring_soon:,.0f}")
print(f"   คิดเป็น: {expiring_pct:.1f}% ของทั้งหมด")
print(f"   Target: < 2%")

🎯 Value Expiring in 30 Days: $13,940,517
   คิดเป็น: 57.8% ของทั้งหมด
   Target: < 2%


In [31]:
# ตรวจสอบช่วงวันที่
print("📅 Expiry_Date:")
print(f"  Min: {df['Expiry_Date'].min()}")
print(f"  Max: {df['Expiry_Date'].max()}")

print("\n📅 Received_Date:")
print(f"  Min: {df['Received_Date'].min()}")
print(f"  Max: {df['Received_Date'].max()}")

print(f"\n📅 วันนี้: {pd.Timestamp.today()}")

# แยกสินค้าตามสถานะ
already_expired = (df['Days_To_Expiry'] < 0).sum()
expiring_30 = ((df['Days_To_Expiry'] >= 0) & (df['Days_To_Expiry'] <= 30)).sum()
still_good = (df['Days_To_Expiry'] > 30).sum()

print(f"\n📦 SKUs ที่หมดอายุแล้ว: {already_expired}")
print(f"📦 SKUs หมดอายุใน 30 วัน: {expiring_30}")
print(f"📦 SKUs ยังไม่ใกล้หมด: {still_good}")

📅 Expiry_Date:
  Min: 2025-03-17 00:00:00
  Max: 2030-07-30 00:00:00

📅 Received_Date:
  Min: 2025-03-13 00:00:00
  Max: 2025-09-09 00:00:00

📅 วันนี้: 2026-04-20 05:47:03.976085

📦 SKUs ที่หมดอายุแล้ว: 495
📦 SKUs หมดอายุใน 30 วัน: 9
📦 SKUs ยังไม่ใกล้หมด: 496


In [32]:
# ===== KPI 2: Value Expiring within 30 Days (แก้ใหม่) =====

today = pd.Timestamp.today()
df['Days_To_Expiry'] = (df['Expiry_Date'] - today).dt.days

# ⭐ แก้: นับเฉพาะ 0-30 วันข้างหน้า (ไม่รวมที่หมดแล้ว)
expiring_soon = df[(df['Days_To_Expiry'] >= 0) & (df['Days_To_Expiry'] <= 30)]

value_expiring = expiring_soon['Total_Inventory_Value_USD'].sum()
total_value = df['Total_Inventory_Value_USD'].sum()
expiring_pct = (value_expiring / total_value) * 100

# แยก: สินค้าที่หมดอายุไปแล้ว (เป็นปัญหาต่างหาก)
already_expired = df[df['Days_To_Expiry'] < 0]
expired_value = already_expired['Total_Inventory_Value_USD'].sum()
expired_pct = (expired_value / total_value) * 100

print("="*50)
print("🎯 KPI 2: Value Expiring within 30 Days")
print("="*50)

print(f"\n📦 สินค้าที่จะหมดอายุใน 30 วัน:")
print(f"   จำนวน SKU: {len(expiring_soon)}")
print(f"   มูลค่า:     ${value_expiring:,.2f}")
print(f"   คิดเป็น:    {expiring_pct:.2f}% ของทั้งหมด")

print(f"\n🚨 ปัญหาเพิ่มเติม: สินค้าที่หมดอายุไปแล้ว!")
print(f"   จำนวน SKU: {len(already_expired)}")
print(f"   มูลค่า:     ${expired_value:,.2f}")
print(f"   คิดเป็น:    {expired_pct:.2f}% ของทั้งหมด")

print(f"\n💡 Interpretation:")
print(f"   Target: < 2%")
if expiring_pct < 2:
    print(f"   ✅ KPI 2 อยู่ในเกณฑ์ปกติ")
else:
    print(f"   ⚠️ KPI 2 เกินเป้า")

🎯 KPI 2: Value Expiring within 30 Days

📦 สินค้าที่จะหมดอายุใน 30 วัน:
   จำนวน SKU: 9
   มูลค่า:     $253,583.08
   คิดเป็น:    1.05% ของทั้งหมด

🚨 ปัญหาเพิ่มเติม: สินค้าที่หมดอายุไปแล้ว!
   จำนวน SKU: 495
   มูลค่า:     $13,686,933.52
   คิดเป็น:    56.76% ของทั้งหมด

💡 Interpretation:
   Target: < 2%
   ✅ KPI 2 อยู่ในเกณฑ์ปกติ


markdown### KPI 3: Weighted Forecast Accuracy

**คำถาม:** พยากรณ์ยอดขายแม่นยำแค่ไหน (ถ่วงน้ำหนักด้วยมูลค่า)?

**สูตร:**
Weighted = Σ(Accuracy × Value) / Σ(Value)

**ทำไม Weighted?**
- SKUs มูลค่าสูง = สำคัญกว่า
- ถ่วงน้ำหนักให้สะท้อนความเป็นจริง

**Target:** ≥ 85%

In [33]:
# ===== KPI 3: Weighted Forecast Accuracy =====

# คำนวณ weighted average
# สูตร: Σ(Accuracy × Value) / Σ(Value)
weighted_accuracy = (
    df['Demand_Forecast_Accuracy_Pct'] * df['Total_Inventory_Value_USD']
).sum() / df['Total_Inventory_Value_USD'].sum()

# เทียบกับ simple average
simple_avg = df['Demand_Forecast_Accuracy_Pct'].mean()

# แสดงผล
print("="*50)
print("🎯 KPI 3: Weighted Forecast Accuracy")
print("="*50)
print(f"\n📊 Weighted Average: {weighted_accuracy:.1f}%")
print(f"📊 Simple Average:   {simple_avg:.1f}%")
print(f"📊 Min:              {df['Demand_Forecast_Accuracy_Pct'].min():.1f}%")
print(f"📊 Max:              {df['Demand_Forecast_Accuracy_Pct'].max():.1f}%")

# ตีความ
print("\n💡 Interpretation:")
print(f"   Target: ≥ 85%")
if weighted_accuracy >= 85:
    print(f"   ✅  พยากรณ์แม่นยำดี!")
elif weighted_accuracy >= 70:
    print(f"   🟡  พยากรณ์ปานกลาง — ควรปรับปรุง")
else:
    print(f"   🚨  พยากรณ์แย่ — ต้องทบทวน model!")

🎯 KPI 3: Weighted Forecast Accuracy

📊 Weighted Average: 78.6%
📊 Simple Average:   79.0%
📊 Min:              60.0%
📊 Max:              98.0%

💡 Interpretation:
   Target: ≥ 85%
   🟡  พยากรณ์ปานกลาง — ควรปรับปรุง


 (Optional) ดูลึกกว่านี้ !!หา SKUs ที่ Forecast แย่ที่สุด ไหมครับ? เพื่อเจาะลึกปัญหา:

In [34]:
# SKUs ที่ Forecast Accuracy แย่ที่สุด
worst_forecast = df.nsmallest(10, 'Demand_Forecast_Accuracy_Pct')[
    ['SKU_ID', 'SKU_Name', 'Category', 'Demand_Forecast_Accuracy_Pct', 'Total_Inventory_Value_USD']
]

print("🚨 Top 10 SKUs ที่พยากรณ์แย่ที่สุด:")
print(worst_forecast)

🚨 Top 10 SKUs ที่พยากรณ์แย่ที่สุด:
      SKU_ID               SKU_Name       Category  \
747  SKU0748      Fresh Product 134  Fresh Produce   
315  SKU0316    Seafood Product 232        Seafood   
943  SKU0944        Meat Product 97           Meat   
348  SKU0349     Bakery Product 275         Bakery   
576  SKU0577     Frozen Product 366         Frozen   
991  SKU0992     Frozen Product 418         Frozen   
492  SKU0493     Bakery Product 128         Bakery   
111  SKU0112    Personal Product 17  Personal Care   
138  SKU0139     Pantry Product 326         Pantry   
716  SKU0717  Household Product 296      Household   

     Demand_Forecast_Accuracy_Pct  Total_Inventory_Value_USD  
747                         60.01                    1.17988  
315                         60.05                21404.00000  
943                         60.09                    2.73033  
348                         60.21                54606.00000  
576                         60.22                    2.

หรือดู Forecast Accuracy แยกตาม Category:

In [35]:
# Forecast Accuracy by Category
print("📊 Forecast Accuracy by Category:")
print(df.groupby('Category')['Demand_Forecast_Accuracy_Pct'].agg(['mean', 'min', 'max']).round(1))

📊 Forecast Accuracy by Category:
               mean   min   max
Category                       
Bakery         80.1  60.2  97.8
Beverages      78.2  60.5  97.8
Dairy          78.6  60.7  97.3
Fresh Produce  79.8  60.0  97.9
Frozen         77.4  60.2  97.5
Household      77.9  60.5  97.3
Meat           80.3  60.1  97.6
Pantry         78.8  60.4  97.8
Personal Care  79.5  60.4  97.5
Seafood        79.7  60.0  98.0


In [36]:
# ========================================
# 🎯 KPI 4: % SKUs Oversold
# ========================================
# คำถาม: สินค้ากี่ % ที่ขายเกินสต็อก?

QTY_ON_HAND = 'Quantity_On_Hand'
QTY_RESERVED = 'Quantity_Reserved'
QTY_COMMITTED = 'Quantity_Committed'

# ตรวจ: Reserved + Committed > On_Hand หรือไม่
df['Oversold'] = (df[QTY_RESERVED] + df[QTY_COMMITTED]) > df[QTY_ON_HAND]

pct_oversold = df['Oversold'].mean() * 100
count_oversold = df['Oversold'].sum()

print(f"🎯 % SKUs Oversold: {pct_oversold:.2f}%")
print(f"   จำนวน SKU: {count_oversold} จาก {len(df)}")
print(f"   Target: 0%")

🎯 % SKUs Oversold: 2.80%
   จำนวน SKU: 28 จาก 1000
   Target: 0%


In [37]:
# ========================================
# 📊 KPI DASHBOARD SUMMARY
# ========================================
print("="*60)
print("🎯 E-GROCERY SUPPLY CHAIN KPI DASHBOARD")
print("="*60)

print(f"\n1. Median Sellable Coverage:  {median_coverage:.1f} วัน  (Target: 7-14 วัน)")
print(f"2. Value Expiring 30 Days:    ${value_expiring:,.0f} ({expiring_pct:.2f}%)  (Target: <2%)")
print(f"3. Weighted Forecast Acc:     {weighted_accuracy:.1f}%  (Target: ≥85%)")
print(f"4. % SKUs Oversold:           {pct_oversold:.2f}%  (Target: 0%)")

print("\n" + "="*60)
print("📋 STATUS SUMMARY")
print("="*60)

# สถานะแต่ละ KPI
kpi_status = [
    ("KPI 1 - Coverage", median_coverage, 7, 14, "วัน"),
    ("KPI 2 - Expiring %", expiring_pct, 0, 2, "%"),
    ("KPI 3 - Forecast %", weighted_accuracy, 85, 100, "%"),
    ("KPI 4 - Oversold %", pct_oversold, 0, 0, "%"),
]

# แสดงสถานะ
print(f"\n✅ KPI 2: Value Expiring 30 Days ({expiring_pct:.2f}%) — PASS")
print(f"⚠️  KPI 1: Sellable Coverage ({median_coverage:.1f} วัน) — BELOW TARGET")
print(f"⚠️  KPI 3: Forecast Accuracy ({weighted_accuracy:.1f}%) — BELOW TARGET")
print(f"🚨 KPI 4: SKUs Oversold ({pct_oversold:.2f}%) — ABOVE TARGET")

print("\n" + "="*60)
print("💡 KEY INSIGHTS")
print("="*60)
print(f"\n🔴 CRITICAL ISSUE: สินค้ามูลค่า $13.69M หมดอายุไปแล้ว (56.76%)")
print(f"🔍 ROOT CAUSE: Forecast Accuracy ต่ำ (78.6%) ทำให้ซื้อเกินจำเป็น")
print(f"🎯 PRIORITY ACTION:")
print(f"   1. ทบทวน Forecasting Model (Frozen category แย่สุด)")
print(f"   2. สร้าง watchlist สำหรับ high-value SKUs (28 ตัวที่ oversold)")
print(f"   3. Implement markdown policy เร่งด่วน")

print("\n" + "="*60)

🎯 E-GROCERY SUPPLY CHAIN KPI DASHBOARD

1. Median Sellable Coverage:  6.2 วัน  (Target: 7-14 วัน)
2. Value Expiring 30 Days:    $253,583 (1.05%)  (Target: <2%)
3. Weighted Forecast Acc:     78.6%  (Target: ≥85%)
4. % SKUs Oversold:           2.80%  (Target: 0%)

📋 STATUS SUMMARY

✅ KPI 2: Value Expiring 30 Days (1.05%) — PASS
⚠️  KPI 1: Sellable Coverage (6.2 วัน) — BELOW TARGET
⚠️  KPI 3: Forecast Accuracy (78.6%) — BELOW TARGET
🚨 KPI 4: SKUs Oversold (2.80%) — ABOVE TARGET

💡 KEY INSIGHTS

🔴 CRITICAL ISSUE: สินค้ามูลค่า $13.69M หมดอายุไปแล้ว (56.76%)
🔍 ROOT CAUSE: Forecast Accuracy ต่ำ (78.6%) ทำให้ซื้อเกินจำเป็น
🎯 PRIORITY ACTION:
   1. ทบทวน Forecasting Model (Frozen category แย่สุด)
   2. สร้าง watchlist สำหรับ high-value SKUs (28 ตัวที่ oversold)
   3. Implement markdown policy เร่งด่วน



In [40]:
import pandas as pd

kpi_summary = pd.DataFrame({
    'kpi_name': ['Sellable Coverage', 'Value Expiring 30D', 'Forecast Accuracy', 'SKUs Oversold'],
    'value': [6.2, 1.05, 78.6, 2.80],
    'target': [14, 2.0, 85, 2.0],
    'unit': ['days', '%', '%', '%'],
    'status': ['BELOW', 'PASS', 'BELOW', 'ABOVE']
})

kpi_summary.to_csv('kpi_summary.csv', index=False)

from google.colab import files
files.download('kpi_summary.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>